# Stage 3 · Modern RLHF Variants — SOLUTION
### Topics: DPO · IPO · SimPO · KTO · GRPO vs PPO · Offline vs Online Tradeoffs


In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math
from typing import List, Tuple, Optional, Dict
from dataclasses import dataclass


---
## 1 · Direct Preference Optimization (DPO)

**DPO (Rafailov et al., 2023)** is the key insight that made offline preference training practical.

### The key derivation
The RLHF objective with KL constraint has a closed-form optimal policy:
$$\pi^*(y|x) = \frac{1}{Z(x)}\pi_{ref}(y|x) \exp\!\left(\frac{r(x,y)}{\beta}\right)$$

Rearranging: the reward can be expressed in terms of the policy and reference:
$$r(x,y) = \beta \log \frac{\pi^*(y|x)}{\pi_{ref}(y|x)} + \beta \log Z(x)$$

Substituting into the Bradley-Terry objective and noting $Z(x)$ cancels:

$$\mathcal{L}_{DPO} = -\mathbb{E}\!\left[\log \sigma\!\left(\beta \left(\log \frac{\pi_\theta(y_w|x)}{\pi_{ref}(y_w|x)} - \log \frac{\pi_\theta(y_l|x)}{\pi_{ref}(y_l|x)}\right)\right)\right]$$

### Why DPO is powerful
- **No reward model needed** — the policy directly parameterises the implicit reward
- **No RL loop** — standard supervised training on preference pairs
- **Stable** — no clipping, no GAE, no value network
- **Tradeoff** — offline, so distribution shift can be an issue; weaker than PPO on hard tasks


In [15]:
def dpo_loss(
    logprobs_policy_chosen:   torch.Tensor,  # (B,) — log π_θ(y_w|x)
    logprobs_policy_rejected: torch.Tensor,  # (B,) — log π_θ(y_l|x)
    logprobs_ref_chosen:      torch.Tensor,  # (B,) — log π_ref(y_w|x), detached
    logprobs_ref_rejected:    torch.Tensor,  # (B,) — log π_ref(y_l|x), detached
    beta: float = 0.1,
) -> Tuple[torch.Tensor, Dict[str, float]]:
    """
    DPO loss = -mean( log σ( β * (log_ratio_w - log_ratio_l) ) )
    where log_ratio = log π_θ(y|x) - log π_ref(y|x)

    Returns (loss, metrics_dict).
    """
    log_ratio_w = logprobs_policy_chosen   - logprobs_ref_chosen.detach()    # (B,)
    log_ratio_l = logprobs_policy_rejected - logprobs_ref_rejected.detach()  # (B,)

    rewards_chosen   = beta * log_ratio_w   # implicit reward for chosen
    rewards_rejected = beta * log_ratio_l   # implicit reward for rejected

    loss = -F.logsigmoid(rewards_chosen - rewards_rejected).mean()

    # Useful diagnostics
    acc = (rewards_chosen > rewards_rejected).float().mean().item()
    margin = (rewards_chosen - rewards_rejected).mean().item()

    return loss, {
        "loss": loss.item(),
        "reward_chosen": rewards_chosen.mean().item(),
        "reward_rejected": rewards_rejected.mean().item(),
        "reward_margin": margin,
        "accuracy": acc,
    }


def dpo_implicit_reward(
    logprobs_policy: torch.Tensor,   # (B,)
    logprobs_ref:    torch.Tensor,   # (B,)
    beta:            float = 0.1,
) -> torch.Tensor:                   # (B,)
    """
    The implicit reward of a completion under DPO:
    r_implicit(x,y) = β * log(π_θ(y|x) / π_ref(y|x))
    """
    return beta * (logprobs_policy - logprobs_ref.detach())


# ── Sanity checks ─────────────────────────────────────────────────────────
torch.manual_seed(0)
B = 8
lp_pol_w = torch.tensor([-1.0, -1.5, -0.8, -2.0, -1.2, -0.9, -1.8, -1.1])
lp_pol_l = torch.tensor([-2.0, -2.5, -1.8, -3.0, -2.2, -1.9, -2.8, -2.1])
lp_ref_w = lp_pol_w.clone()  # policy starts at reference
lp_ref_l = lp_pol_l.clone()

# At init (policy==ref): log_ratio = 0, loss = log(2)
loss_init, info_init = dpo_loss(lp_pol_w, lp_pol_l, lp_ref_w, lp_ref_l, beta=0.1)
assert abs(loss_init.item() - math.log(2)) < 1e-5,     f"At init, DPO loss should equal log(2)={math.log(2):.4f}, got {loss_init.item():.4f}"

# After optimisation: chosen log-ratio > rejected log-ratio → loss decreases, acc → 1
lp_pol_w_opt = lp_ref_w + 1.0   # increase log π_θ(y_w)
lp_pol_l_opt = lp_ref_l - 1.0   # decrease log π_θ(y_l)
loss_opt, info_opt = dpo_loss(lp_pol_w_opt, lp_pol_l_opt, lp_ref_w, lp_ref_l, beta=0.1)
assert loss_opt.item() < loss_init.item(), "Optimised loss should be lower"
assert info_opt["accuracy"] == 1.0,        "Perfect separation should give accuracy=1"

print(f"dpo_loss ✓")
print(f"  Init:      loss={loss_init.item():.4f}  acc={info_init['accuracy']:.2f}  margin={info_init['reward_margin']:.4f}")
print(f"  Optimised: loss={loss_opt.item():.4f}  acc={info_opt['accuracy']:.2f}   margin={info_opt['reward_margin']:.4f}")


dpo_loss ✓
  Init:      loss=0.6931  acc=0.00  margin=0.0000
  Optimised: loss=0.5981  acc=1.00   margin=0.2000


---
## 2 · DPO Variants: IPO, KTO, SimPO

Each variant fixes a known failure mode of DPO.

### IPO (Identity Preference Optimization, Azar et al., 2023)
DPO can overfit — the log-ratio can grow without bound if the model is unconstrained.  
IPO uses an **L2 regulariser** instead of log-sigmoid:

$$\mathcal{L}_{IPO} = \mathbb{E}\!\left[\left(\log \frac{\pi(y_w|x)}{\pi_{ref}(y_w|x)} - \log \frac{\pi(y_l|x)}{\pi_{ref}(y_l|x)} - \frac{1}{2\tau}\right)^2\right]$$

Target margin is $1/(2\tau)$; the policy is penalised for exceeding it.

### KTO (Kahneman-Tversky Optimization, Ethayarajh et al., 2024)
Works on **unpaired** data (just "good" or "bad" completions, no pairs).  
Uses prospect theory: losses feel larger than equivalent gains (loss aversion).

$$\mathcal{L}_{KTO} = \mathbb{E}\left[\lambda_w \cdot \sigma(\hat{r}_w - z_0) + \lambda_l \cdot \sigma(z_0 - \hat{r}_l)\right]$$

where $\hat{r} = \beta(\log \pi_\theta(y|x) - \log \pi_{ref}(y|x))$ and $z_0$ is an estimated KL baseline.

### SimPO (Simple Preference Optimization, Meng et al., 2024)
Eliminates the reference model entirely. Uses **length-normalised** log-probs and a **margin** γ:

$$\mathcal{L}_{SimPO} = -\mathbb{E}\!\left[\log \sigma\!\left(\frac{\beta}{|y_w|}\log \pi(y_w|x) - \frac{\beta}{|y_l|}\log \pi(y_l|x) - \gamma\right)\right]$$

Length normalisation prevents the model from preferring short completions.


In [16]:
def ipo_loss(
    logprobs_policy_chosen:   torch.Tensor,  # (B,)
    logprobs_policy_rejected: torch.Tensor,  # (B,)
    logprobs_ref_chosen:      torch.Tensor,  # (B,)  detached
    logprobs_ref_rejected:    torch.Tensor,  # (B,)  detached
    tau: float = 0.1,
) -> torch.Tensor:
    """
    IPO: L2 penalty on log-ratio difference, target margin = 1/(2τ).
    ((log_ratio_w - log_ratio_l) - 1/(2τ))²
    """
    log_ratio_w = logprobs_policy_chosen   - logprobs_ref_chosen.detach()
    log_ratio_l = logprobs_policy_rejected - logprobs_ref_rejected.detach()
    target      = 1.0 / (2 * tau)
    return ((log_ratio_w - log_ratio_l) - target).pow(2).mean()


def simpo_loss(
    logprobs_policy_chosen:   torch.Tensor,   # (B,) — sum of log-probs over sequence
    logprobs_policy_rejected: torch.Tensor,   # (B,)
    len_chosen:   torch.Tensor,               # (B,) — length of chosen completion
    len_rejected: torch.Tensor,               # (B,)
    beta:  float = 2.5,
    gamma: float = 0.5,
) -> Tuple[torch.Tensor, float]:
    """
    SimPO: no reference model. Length-normalised log-probs + margin γ.
    L = -mean( log σ( β/|yw|·logπ(yw) - β/|yl|·logπ(yl) - γ ) )
    """
    # Length-normalised log-probs (avoid division by zero)
    norm_w = logprobs_policy_chosen   / len_chosen.clamp(min=1).float()
    norm_l = logprobs_policy_rejected / len_rejected.clamp(min=1).float()

    logit  = beta * norm_w - beta * norm_l - gamma
    loss   = -F.logsigmoid(logit).mean()
    acc    = (logit > 0).float().mean().item()
    return loss, acc


def kto_loss(
    logprobs_policy: torch.Tensor,   # (B,) — log π_θ(y|x) for ALL responses (chosen AND rejected mixed)
    logprobs_ref:    torch.Tensor,   # (B,)  detached
    is_good:         torch.Tensor,   # (B,)  bool — True if this response is "desirable"
    beta:  float = 0.1,
    lam_w: float = 1.0,   # weight for desirable completions
    lam_l: float = 1.0,   # weight for undesirable completions
) -> torch.Tensor:
    """
    KTO: works on unpaired data. No pairwise comparisons needed.
    Uses the mean log-ratio as the KL baseline z_0.
    L = -E_w[ λ_w σ(r_w - z_0) ] - E_l[ λ_l σ(z_0 - r_l) ]
    """
    r_hat = beta * (logprobs_policy - logprobs_ref.detach())  # (B,) implicit rewards

    # KL baseline: mean implicit reward over the batch
    z_0   = r_hat.detach().mean()

    loss_good = -lam_w * F.logsigmoid(r_hat[is_good]  - z_0).mean()  if is_good.any()  else torch.tensor(0.0)
    loss_bad  = -lam_l * F.logsigmoid(z_0 - r_hat[~is_good]).mean()  if (~is_good).any() else torch.tensor(0.0)

    return loss_good + loss_bad


# ── Sanity checks ─────────────────────────────────────────────────────────
torch.manual_seed(0)
B = 8
lp_pw = torch.randn(B, requires_grad=True)
lp_pl = torch.randn(B, requires_grad=True)
lp_rw = lp_pw.detach() - 0.5
lp_rl = lp_pl.detach() + 0.5

# IPO: loss = 0 when log-ratio difference == 1/(2τ)
log_diff_target = 1.0 / (2 * 0.1)   # = 5.0
lp_pw_t = lp_rw + log_diff_target
lp_pl_t = lp_rl
ipo = ipo_loss(lp_pw_t, lp_pl_t, lp_rw, lp_rl, tau=0.1)
assert ipo.item() < 1e-8, f"IPO loss at target should be 0, got {ipo.item()}"

# SimPO: runs and returns scalar
len_w = torch.randint(5, 20, (B,)).long()
len_l = torch.randint(5, 20, (B,)).long()
lp_pw_detach = lp_pw.detach()
lp_pl_detach = lp_pl.detach()
s_loss, s_acc = simpo_loss(lp_pw_detach, lp_pl_detach, len_w, len_l)
assert s_loss.shape == ()

# KTO: runs on mixed pool
lp_all  = torch.randn(B)
lp_rall = torch.randn(B)
is_good = torch.tensor([True, True, False, True, False, False, True, False])
kto = kto_loss(lp_all, lp_rall, is_good)
assert kto.shape == ()

print(f"ipo_loss   ✓  loss at target={ipo.item():.2e}")
print(f"simpo_loss ✓  loss={s_loss.item():.4f}  acc={s_acc:.2f}")
print(f"kto_loss   ✓  loss={kto.item():.4f}")


ipo_loss   ✓  loss at target=0.00e+00
simpo_loss ✓  loss=0.9529  acc=0.00
kto_loss   ✓  loss=1.4050


---
## 3 · GRPO vs PPO — Deep Comparison

Both use the PPO clipped objective, but they differ fundamentally in how advantages are computed.

### PPO (online, with critic)
```
For each batch:
  1. Collect rollouts with π_θ
  2. Compute V(s_t) using value network V_φ
  3. Compute GAE: δ_t = r_t + γV(s_{t+1}) - V(s_t), Â_t = Σ (γλ)^k δ_{t+k}
  4. Update π_θ with clipped surrogate
  5. Update V_φ with MSE loss on returns
```

### GRPO (online, no critic)
```
For each batch of prompts:
  1. Sample G completions per prompt
  2. Score each with reward function → R_i
  3. Normalise within group: A_i = (R_i - mean_G) / std_G
  4. Update π_θ with clipped surrogate using A_i
```

### Key differences

| Aspect | PPO | GRPO |
|---|---|---|
| Value network | Yes (same backbone, extra head) | **No** |
| Advantage estimator | GAE (bootstrapped) | Group mean (MC within group) |
| Memory | 2× model memory | 1× model memory |
| Bias | Lower (bootstrapping) | Higher (MC) |
| Variance | Higher (bootstrap noise) | Lower (group normalisation) |
| Sample efficiency | Higher (reuse rollouts) | Lower (fresh samples per prompt) |
| Stability | Requires careful V_φ tuning | More robust, fewer hyperparams |

### When to use each
- **PPO:** complex tasks with dense per-step rewards (games, robotics, multi-turn dialogue)
- **GRPO:** terminal reward tasks (math correctness, coding, summarisation) — especially at scale


In [17]:
def ppo_advantage_with_critic(
    rewards:     torch.Tensor,  # (B, T) — per-token rewards
    values:      torch.Tensor,  # (B, T) — V(s_t), detached
    next_values: torch.Tensor,  # (B, T) — V(s_{t+1}), 0 at terminal, detached
    comp_mask:   torch.Tensor,  # (B, T) — 1 for completion tokens
    gamma: float = 0.99,
    lam:   float = 0.95,
) -> Tuple[torch.Tensor, torch.Tensor]:   # (advantages, returns) both (B, T)
    """
    GAE advantage for token-level PPO in LLMs.
    Computed independently per sequence (mask handles padding).
    Returns (advantages, returns_for_value_training).
    """
    B, T    = rewards.shape
    adv     = torch.zeros_like(rewards)
    returns = torch.zeros_like(rewards)
    gae     = torch.zeros(B)

    for t in reversed(range(T)):
        mask_t   = comp_mask[:, t].float()
        delta    = rewards[:, t] + gamma * next_values[:, t] - values[:, t]
        gae      = (delta + gamma * lam * gae) * mask_t
        adv[:, t] = gae
        returns[:, t] = adv[:, t] + values[:, t]

    return adv, returns


def grpo_advantage(
    rewards: torch.Tensor,   # (B*G,) — scalar reward per completion
    G: int,
    eps: float = 1e-8,
) -> torch.Tensor:           # (B*G,) — normalised advantages
    """Group-relative normalisation within each group of G completions."""
    rewards_g = rewards.view(-1, G)
    adv = (rewards_g - rewards_g.mean(dim=1, keepdim=True)) /           (rewards_g.std(dim=1, keepdim=True) + eps)
    return adv.view(-1)


def compare_advantage_variance(G: int = 8, n_trials: int = 1000) -> Dict[str, float]:
    """
    Empirically compare variance of GRPO advantages vs raw rewards.
    GRPO normalisation should reduce cross-group variance.
    """
    torch.manual_seed(0)
    # Simulate rewards where group means differ (common in practice)
    group_means = torch.randn(n_trials) * 2.0          # different difficulty per prompt
    raw_rewards = group_means.repeat_interleave(G) + torch.randn(n_trials * G) * 0.5

    grpo_adv    = grpo_advantage(raw_rewards, G=G)

    return {
        "raw_reward_std":  raw_rewards.std().item(),
        "grpo_adv_std":    grpo_adv.std().item(),
        "raw_reward_mean": raw_rewards.mean().item(),
        "grpo_adv_mean":   grpo_adv.mean().item(),
    }


# ── Sanity checks ─────────────────────────────────────────────────────────
torch.manual_seed(0)
B, T, G = 4, 12, 4
rewards  = torch.randn(B, T) * 0.1
values   = torch.randn(B, T)
nxt_vals = torch.cat([values[:, 1:].detach(), torch.zeros(B, 1)], dim=1)
comp_mask = torch.cat([torch.zeros(B, 4), torch.ones(B, 8)], dim=1).long()

adv, rets = ppo_advantage_with_critic(rewards, values.detach(), nxt_vals.detach(), comp_mask)
assert adv.shape == (B, T) and rets.shape == (B, T)
assert (adv[:, :4] == 0).all(), "Prompt positions should have zero advantage"

grpo_adv = grpo_advantage(torch.randn(B * G), G=G)
assert grpo_adv.shape == (B * G,)
# GRPO advantages are per-group normalised: each group of G has mean≈0, std≈1
for i in range(B):
    g = grpo_adv[i*G:(i+1)*G]
    assert g.mean().abs() < 1e-4

stats = compare_advantage_variance(G=8)
print(f"ppo_advantage_with_critic ✓  adv range: [{adv.min():.3f}, {adv.max():.3f}]")
print(f"grpo_advantage            ✓")
print(f"Variance comparison:")
for k, v in stats.items():
    print(f"  {k:22s}: {v:.4f}")


ppo_advantage_with_critic ✓  adv range: [-3.380, 1.800]
grpo_advantage            ✓
Variance comparison:
  raw_reward_std        : 2.1122
  grpo_adv_std          : 0.9355
  raw_reward_mean       : 0.0494
  grpo_adv_mean         : -0.0000


---
## 4 · Offline vs Online — Distribution Shift

### Offline methods (DPO, IPO, SimPO)
- Train on a **fixed dataset** of human preferences
- No new generations during training
- Risk: **distribution shift** — the policy may move far from the data-generating distribution,  
  making the preference labels unreliable (they were collected under a different model)
- Mitigation: **iterative DPO** — periodically generate new completions and collect new labels

### Online methods (PPO, GRPO)
- Continuously generate new completions and score them
- Always on-policy (or near-policy with clipping)
- More expensive but avoids distribution shift by design
- **On-policy ratio:** how close are current rollouts to the training distribution?

### Measuring distribution shift
The probability ratio $\rho = \pi_\theta(y|x) / \pi_{data}(y|x)$ measures how much the current policy  
differs from the data-generating policy. PPO clips this; DPO has no such mechanism.

### The offline-online spectrum

| Method | Data | Updates | Stability | Performance |
|---|---|---|---|---|
| SFT | Static | Offline | High | Baseline |
| DPO | Static preference pairs | Offline | High | Good |
| Iterative DPO | Refreshed periodically | Semi-online | Medium | Better |
| GRPO | Generated on-the-fly | Online | Medium | Strong |
| PPO | Generated on-the-fly | Online | Lower | Strongest |


In [18]:
def importance_weight(
    logprobs_policy:    torch.Tensor,  # (B,) — log π_θ(y|x)
    logprobs_behaviour: torch.Tensor,  # (B,) — log π_data(y|x)  (data-generating policy)
    clip_ratio: float = 5.0,
) -> torch.Tensor:                     # (B,)
    """
    Importance weight ρ = π_θ / π_data, clipped to [1/clip_ratio, clip_ratio].
    Used in off-policy corrections.
    """
    log_rho = logprobs_policy - logprobs_behaviour.detach()
    rho     = torch.exp(log_rho)
    return torch.clamp(rho, 1.0 / clip_ratio, clip_ratio)


def dpo_with_sft_regularizer(
    logprobs_policy_chosen:   torch.Tensor,   # (B,)
    logprobs_policy_rejected: torch.Tensor,   # (B,)
    logprobs_ref_chosen:      torch.Tensor,   # (B,)  detached
    logprobs_ref_rejected:    torch.Tensor,   # (B,)  detached
    beta:     float = 0.1,
    sft_coef: float = 0.1,
) -> Tuple[torch.Tensor, Dict[str, float]]:
    """
    DPO + SFT regularizer: add NLL loss on chosen completions to prevent drift.
    L = L_DPO + sft_coef * (-mean(logp_policy_chosen))
    This helps prevent distribution shift in offline training.
    """
    log_ratio_w = logprobs_policy_chosen   - logprobs_ref_chosen.detach()
    log_ratio_l = logprobs_policy_rejected - logprobs_ref_rejected.detach()
    dpo_l = -F.logsigmoid(beta * (log_ratio_w - log_ratio_l)).mean()
    sft_l = -logprobs_policy_chosen.mean()
    total = dpo_l + sft_coef * sft_l
    return total, {"dpo": dpo_l.item(), "sft": sft_l.item(), "total": total.item()}


def estimate_distribution_shift(
    logprobs_current: torch.Tensor,   # (B,) — log π_θ(y|x)
    logprobs_data:    torch.Tensor,   # (B,) — log π_data(y|x)
) -> Dict[str, float]:
    """
    Diagnostics for distribution shift:
    - mean log ratio (positive = policy prefers these sequences more than data did)
    - effective sample size (ESS) = (Σ ρ_i)² / Σ ρ_i²  ∈ (0, B]
      ESS close to B = similar weights (e.g. all ρ equal); ESS << B = few samples dominate.
      Note: a uniform log-ratio (same +c on every row) makes all ρ equal → ESS = B.
    """
    log_rho = logprobs_current - logprobs_data.detach()
    rho     = torch.exp(log_rho)
    ess     = rho.sum().pow(2) / rho.pow(2).sum()
    return {
        "mean_log_ratio": log_rho.mean().item(),
        "max_log_ratio":  log_rho.abs().max().item(),
        "ess":            ess.item(),
        "ess_fraction":   (ess / len(rho)).item(),
    }


# ── Sanity checks ─────────────────────────────────────────────────────────
torch.manual_seed(0)
B = 16
lp_pol  = torch.randn(B, requires_grad=True)
lp_data = torch.randn(B)

# Same policy and data → IW = 1, ESS = B
iw_same = importance_weight(lp_data, lp_data)
assert iw_same.mean().abs().item() - 1.0 < 1e-5

shift_same = estimate_distribution_shift(lp_data, lp_data)
assert abs(shift_same["ess_fraction"] - 1.0) < 1e-5, "No shift → ESS fraction = 1"

# Spread log-importance weights → low ESS (uniform +c keeps all ρ equal → ESS = B).
lp_shifted = lp_data + torch.linspace(0.0, 6.0, B, device=lp_data.device, dtype=lp_data.dtype)
shift_large = estimate_distribution_shift(lp_shifted, lp_data)
assert shift_large["ess_fraction"] < 0.5, "Large shift should reduce ESS"

lp_pw = torch.randn(B, requires_grad=True)
lp_pl = torch.randn(B)
lp_rw = lp_pw.detach(); lp_rl = lp_pl.detach()
total, info = dpo_with_sft_regularizer(lp_pw, lp_pl, lp_rw, lp_rl)
total.backward()

print(f"importance_weight         ✓  mean IW (same)={iw_same.mean().item():.3f}")
print(f"estimate_distribution_shift ✓")
print(f"  No shift:    ESS={shift_same['ess_fraction']:.3f}")
print(f"  Large shift: ESS={shift_large['ess_fraction']:.3f}")
print(f"dpo_with_sft_regularizer  ✓  {info}")


importance_weight         ✓  mean IW (same)=1.000
estimate_distribution_shift ✓
  No shift:    ESS=1.000
  Large shift: ESS=0.316
dpo_with_sft_regularizer  ✓  {'dpo': 0.6931471824645996, 'sft': -0.06439126282930374, 'total': 0.6867080330848694}


---
## 5 · Algorithm Selection & Practical Comparison

Putting it all together — when to use which algorithm.

### Decision tree
```
Do you have a verifiable reward function (math, code)?
  Yes → GRPO or PPO (online RL)
  No  → Do you have preference pairs?
          Yes → DPO family (offline)
                  Are you worried about overfitting / reward margin?
                    Yes → IPO
                  Is your data unpaired?
                    Yes → KTO
                  Do you have no reference model?
                    Yes → SimPO
          No  → Need to collect preference data first
```

### Key hyperparameters and their effects
| Hyperparameter | Too low | Too high |
|---|---|---|
| β (KL weight) | Reward hacking | Too conservative, no learning |
| ε (PPO clip) | Under-update | Large policy updates, instability |
| G (GRPO groups) | High variance advantages | Expensive (G × forward passes) |
| γ (discount) | Ignores future | Slow convergence, instability |
| λ (GAE) | High bias | High variance |


In [19]:
def run_algorithm_comparison(
    n_steps: int = 50,
    B: int = 16,
    G: int = 4,
    seed: int = 42,
) -> Dict[str, List[float]]:
    """
    Toy comparison: simulate DPO vs GRPO training dynamics on a synthetic task.
    Tracks reward margin (higher = better) and KL divergence from reference.
    """
    torch.manual_seed(seed)

    # Shared initial policy parameters (scalar for simplicity)
    # Simulated as: policy logprob = theta * feature(x,y)
    theta_dpo  = torch.tensor([0.0], requires_grad=True)
    theta_grpo = torch.tensor([0.0], requires_grad=True)
    opt_dpo    = torch.optim.SGD([theta_dpo],  lr=0.1)
    opt_grpo   = torch.optim.SGD([theta_grpo], lr=0.1)

    dpo_margins, grpo_rewards = [], []

    for step in range(n_steps):
        # ── DPO step (offline) ───────────────────────────────────────────
        # Simulate: chosen features are positive, rejected are negative
        feat_w = torch.randn(B).abs()       # (B,) positive features
        feat_l = -torch.randn(B).abs()      # (B,) negative features
        lp_pol_w = theta_dpo * feat_w
        lp_pol_l = theta_dpo * feat_l
        lp_ref_w = feat_w.detach() * 0     # ref at 0
        lp_ref_l = feat_l.detach() * 0

        loss_dpo, info_dpo = dpo_loss(lp_pol_w, lp_pol_l, lp_ref_w, lp_ref_l, beta=0.1)
        opt_dpo.zero_grad()
        loss_dpo.backward()
        opt_dpo.step()
        dpo_margins.append(info_dpo["reward_margin"])

        # ── GRPO step (online) ───────────────────────────────────────────
        # Sample G completions, score with true reward = theta * noise
        features   = torch.randn(B * G)
        rewards_g  = (theta_grpo.detach() * features + features).view(-1)   # proxy reward
        adv        = grpo_advantage(rewards_g, G=G)
        # Log-probs for current policy
        lp_new     = theta_grpo * features
        lp_old     = lp_new.detach()
        loss_grpo  = -torch.min(
            torch.exp(lp_new - lp_old) * adv,
            torch.clamp(torch.exp(lp_new - lp_old), 0.8, 1.2) * adv
        ).mean()
        opt_grpo.zero_grad()
        loss_grpo.backward()
        opt_grpo.step()
        grpo_rewards.append(rewards_g.mean().item())

    return {
        "dpo_reward_margin": dpo_margins,
        "grpo_reward":       grpo_rewards,
        "final_theta_dpo":   theta_dpo.item(),
        "final_theta_grpo":  theta_grpo.item(),
    }


results = run_algorithm_comparison(n_steps=50)
print("Algorithm comparison ✓")
print(f"  DPO  final theta={results['final_theta_dpo']:.4f}  "
      f"final margin={results['dpo_reward_margin'][-1]:.4f}")
print(f"  GRPO final theta={results['final_theta_grpo']:.4f}  "
      f"final reward={results['grpo_reward'][-1]:.4f}")
assert results['final_theta_dpo'] > 0,  "DPO should push theta positive"
assert results['dpo_reward_margin'][-1] > results['dpo_reward_margin'][0],     "DPO reward margin should increase"
print("\nSummary: DPO optimises implicit reward margin; GRPO optimises group-relative reward.")


Algorithm comparison ✓
  DPO  final theta=0.3998  final margin=0.0738
  GRPO final theta=3.4017  final reward=-0.2117

Summary: DPO optimises implicit reward margin; GRPO optimises group-relative reward.
